01 . Filtering
===
**Pre cleaning of microbiome data** 
Preparing 16S data for further analysis such as beta diversity and statistical tests. 
1. add variabel types to meta data
2. filter features/samples that will not be part of some analysis


In [ ]:
library(phyloseq)
library(stringr)
library(ggplot2)
source("src/microfiltR/microfiltR_source_code.R")

**Load QIIME2 based taxonomy from phyloseq object (rds file)**

In [ ]:
rds <- readRDS(snakemake@input[[1]])
#rds <- readRDS("../../../../../data/tmp/microbiome/quant/qiime2/silva/physeq.rds")
df <- data.frame(as(sample_data(rds), "matrix"))
head(df)

In [ ]:
MODEL = snakemake@config$models[[snakemake@wildcards$model]]

In [ ]:
subset.ids <- function(rds, params){
    df <- data.frame(as(sample_data(rds), "matrix"))
    # set defaults if missing params
    if (!"axis" %in% names(params)) params$axis <- "column"
    if (params$axis == "column"){
        df <- data.frame(as(sample_data(rds), "matrix"))
    } else{
        df <- data.frame(as(tax_table(rds), "matrix"))
    }
    if (!"name" %in% names(params)) params$name <- "Subset"
    if (!params$name %in% colnames(df)) stop("params.name not in df")
    if (!"selection" %in% names(params)) params$selection <- "keep"
    if ("ids" %in% names(params)){
        keep <- sample_names(rds) %in% params$keep
        if (sum(keep) == 0) stop("no overlap in subset ids from model.yaml")
    } else{
        if (!params$name %in% colnames(df)){
            stop(cat("missing column:", params$name, "in metadata."))
        } 
        subset_col <- as.character(df[,params$name])
        keep <- subset_col == as.character(params$selection)
        if (sum(keep) == 0) stop(paste0("no overlap in selection ", params$selection, ": ", params$name))
    }
    if (params$axis == "column"){
        rds <- prune_samples(keep, rds)
    } else{
        rds <- prune_taxa(keep, rds)
    }
}

In [ ]:
if (!is.null(MODEL$subsets)){
    for (p in MODEL$subsets){
        rds <- subset.ids(rds, p)
    }
}
rds

**Taxonomy requirements**  
We require taxa to be from the Bacteria kingdom and remove mithochondria/chloroplasts

In [ ]:
rds.f <- subset_taxa(rds,
    Kingdom == "Bacteria" &
    Family  != "mitochondria" &
    Class   != "Chloroplast" &
    Phylum != "Cyanobacteria/Chloroplast"
  )
rds.f = prune_taxa(taxa_sums(rds.f) > 0, rds.f)
rds.f


**Independent feature filtes**  
Functions for paramter tuning independent filtering is from Bryan Brown's microfiltR https://github.com/itsmisterbrown/microfiltR

In [ ]:
as.threshold <- estimate.ASthreshold(ps=rds.f, WST=NULL, minLIB=500, Prange = c(0.025:0.15, 0.025), CVrange = c(1:10, 0.5), RArange = c(5e-6:0.9e-3, 1e-5))

In [ ]:
e3.f <- microfilter(ps=rds.f, WST=NULL, 
                    PFT=snakemake@params$prevalence_threshold,
                    RAT=snakemake@params$abundance_threshold , 
                    minLIB=snakemake@params$min_lib, 
                    return.all = TRUE)
e3.f$filtered.phyloseq

----------------

**Save filtered SILVA**

In [ ]:
saveRDS(e3.f$filtered.phyloseq, snakemake@output[[1]])
cat("Saved ", snakemake@output[[1]])

--------

--------------
